# 01 · Badminton — odsłuch A/B i pojedyncze uderzenia
**Dane rzeczywiste:** cztery serie central/off-center z jednej rakiety, 46 automatycznych kandydatów.
To demonstracja narzędzi, nie pomiar skuteczności wykrywania wad ceramiki.

Wybierz kernel `.venv` / Ping Lab i **Run All**. Ten notebook działa samodzielnie;
nie musisz wcześniej uruchamiać pozostałych. Granice serii i zapowiedzi pochodzą
z `data/raw/badminton/session.json`. `project.json` pozostaje konfiguracją oryginalnych notebooków.
Nowe wyniki trafiają do osobnego katalogu; nagranie źródłowe pozostaje bez zmian.


In [1]:
NOTEBOOK = '01_pierwsza_para'
from pathlib import Path
import sys, json, hashlib, io, base64
from datetime import datetime
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/pinglab').is_dir()), None)
assert ROOT is not None, 'Otwórz notebook z folderu projektu ceramic-ping-lab.'
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
%matplotlib inline
cfg = json.loads((ROOT/'data/raw/badminton/session.json').read_text())
SOURCE = ROOT/'data/raw/badminton'/cfg['audio']
assert SOURCE.is_file(), 'Brak nagrania badminton: nie przełączamy się automatycznie na demo.'
OUT = ROOT/'reports/runs/badminton-notebooks'/NOTEBOOK/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Audio, HTML, Markdown
from pinglab.audio import load_audio, Config
from pinglab.recorder import segment_ringdown, ringdown_plot
from pinglab.features import extract_features, feature_columns, decay_curve
from pinglab.learning import make_classifier
from pinglab.series import evaluate_series, paired_folds
from pinglab.embeddings import ASTEncoder
from pinglab.robustness import mix_noise

RUN_AST = False
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})
colors = {'central':'#137f83', 'offcenter':'#c56838'}
r = load_audio(SOURCE, Config(sr=48000, max_seconds=180))
print('Plik:', SOURCE.name, '| SHA256:', hashlib.sha256(SOURCE.read_bytes()).hexdigest())
display(pd.DataFrame([r.qc]).drop(columns=['warnings']))
print('Uwagi QC:', r.qc['warnings'] or 'brak')

Plik: session.wav | SHA256: afc2d308b32b80ead3f80f795707a2059d2e359a33a2d6c20470464d41df7f07


,duration_s,native_sr,channels,selected_channel,peak_dbfs,rms_dbfs,clipping_fraction,dc_offset
0,88.99771,44100,1,0,-17.976912,-45.595115,0.0,8.182004e-09


Uwagi QC: ['Pasmo źródłowe kończy się przy 22050 Hz; resampling go nie odzyska']


## Przygotowanie uderzeń
Ten sam detektor i okna: 25 ms przed + 350 ms po początku, skrócone przed kolejnym zdarzeniem. Etykieta pochodzi z serii. Zdarzenie około 77 s pozostaje niepewne.

In [2]:
rows=[]; clips={}; series_traces={}
for s in cfg['series']:
 z=r.y[int(s['start_s']*r.sr):int(s['end_s']*r.sr)]
 tab,t,db=segment_ringdown(z,r.sr,**cfg['detector'])
 series_traces[s['series_id']]=(z,tab,t,db)
 interactive=ringdown_plot(z,r.sr,tab,t,db)
 interactive.write_html(OUT/(s['series_id']+'_segmentacja.html'),include_plotlyjs=True)
 for k,row in enumerate(tab.itertuples()):
  onset=float(row.onset_s);absolute=s['start_s']+onset
  a=max(0,onset-cfg['pre_s'])
  nxt=float(tab.iloc[k+1].onset_s) if k+1<len(tab) else len(z)/r.sr
  b=min(onset+cfg['feature_tail_s'],nxt-.015,len(z)/r.sr)
  clip=z[int(a*r.sr):int(b*r.sr)]
  eid=f"{s['series_id']}_{k+1:02d}"
  clips[eid]=clip
  uncertain=any(abs(absolute-v)<.05 for v in cfg['uncertain_onsets_s'])
  rows.append(dict(event_id=eid,series_id=s['series_id'],round=s['round'],label=s['label'],
   onset_s=absolute,offset_s=onset-a,window_s=len(clip)/r.sr,
   ringdown_s=row.duration_s,ringdown_status=row.status,uncertain=uncertain,
   peak_dbfs=20*np.log10(max(abs(clip).max(),1e-12)),
   **extract_features(clip,r.sr,onset-a,cfg['fmax'])))
events=pd.DataFrame(rows)
events.to_csv(OUT/'features.csv',index=False)
display(events.groupby(['series_id','label'],sort=False).agg(uderzenia=('event_id','size'),
 szczyt_mediana_dbfs=('peak_dbfs','median'),niepewne=('uncertain','sum')))
display(events[['event_id','onset_s','ringdown_s','ringdown_status','uncertain']])
assert all(not (a<=v<=b) for v in events.onset_s for a,b in cfg['speech_excluded_s'])
assert events.window_s.min()>.05

,,uderzenia,szczyt_mediana_dbfs,niepewne
series_id,label,,,
central_1,central,14,-26.799461,0
offcenter_1,offcenter,12,-34.025085,0
central_2,central,10,-31.644835,0
offcenter_2,offcenter,10,-31.444176,1


,event_id,onset_s,ringdown_s,ringdown_status,uncertain
0,central_1_01,8.525,0.070,zanik zaobserwowany,False
1,central_1_02,9.670,0.100,zanik zaobserwowany,False
2,central_1_03,10.860,0.140,zanik zaobserwowany,False
3,central_1_04,11.885,0.315,zanik zaobserwowany,False
4,central_1_05,12.945,0.170,zanik zaobserwowany,False
5,central_1_06,14.110,1.055,kolejne uderzenie przed zanikiem,False
6,central_1_07,15.165,0.570,zanik zaobserwowany,False
7,central_1_08,16.300,0.215,zanik zaobserwowany,False
8,central_1_09,17.385,0.120,zanik zaobserwowany,False
9,central_1_10,18.610,0.115,zanik zaobserwowany,False


## Wybierz parę
Możesz zmienić ID i uruchomić tę komórkę ponownie. Pierwszy odsłuch zachowuje głośność, drugi wyrównuje szczyty, aby łatwiej porównać barwę.

In [3]:
EVENT_A = 'central_1_01'
EVENT_B = 'offcenter_1_01'
from pinglab.plots import compare_hits
selected = events.set_index('event_id').loc[[EVENT_A,EVENT_B]]
display(selected[['series_id','label','onset_s','peak_dbfs','uncertain']])
for normalize in [False,True]:
 display(Markdown('**Wyrównane szczyty**' if normalize else '**Oryginalna głośność**'))
 for eid in [EVENT_A,EVENT_B]:
  print(eid);display(Audio(clips[eid],rate=r.sr,normalize=normalize))
fig=compare_hits([clips[EVENT_A],clips[EVENT_B]],r.sr,[EVENT_A,EVENT_B],
                 selected.offset_s.tolist(),cfg['fmax'])
fig.show(renderer='plotly_mimetype')
fig.write_html(OUT/'porownanie_AB.html',include_plotlyjs=True)
keys=['peak_hz','centroid_hz','bandwidth_hz','decay10_s','late_early_ratio','flatness']
display(selected[keys].T.round(3))


,series_id,label,onset_s,peak_dbfs,uncertain
event_id,,,,,
central_1_01,central_1,central,8.525,-30.197828,False
offcenter_1_01,offcenter_1,offcenter,33.905,-27.026358,False


**Oryginalna głośność**

central_1_01


offcenter_1_01


**Wyrównane szczyty**

central_1_01


offcenter_1_01


event_id,central_1_01,offcenter_1_01
peak_hz,890.625,890.625
centroid_hz,1000.047,1229.359
bandwidth_hz,1384.716,1216.831
decay10_s,0.040,0.030
late_early_ratio,0.006,0.002
flatness,0.014,0.009


**Do opowiedzenia:** czy różnica wynika z samej głośności, czy zostaje także po wyrównaniu poziomów? Wybrane dwa uderzenia ilustrują różnicę; nie są oceną klasyfikatora.

In [4]:
print('GOTOWE:', NOTEBOOK, '| pliki wynikowe:', OUT)

GOTOWE: 01_pierwsza_para | pliki wynikowe: /Users/karolsii/dev/ceramic-ping-lab/reports/runs/badminton-notebooks/01_pierwsza_para/20260928_144849_988678
